In [ ]:
from pathlib import Path
from collections import OrderedDict
import gc
import math
import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.metrics import f1_score, balanced_accuracy_score

# =========================================================
# 0) Reproducibility
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1) Paths
# =========================================================
SPLIT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD")
TRAIN_ROOT = SPLIT_ROOT / "TRAIN"
VAL_ROOT   = SPLIT_ROOT / "VAL"
TEST_ROOT  = SPLIT_ROOT / "TEST"

CKPT_DIR = SPLIT_ROOT / "CNN_CKPTS_FOR_BILSTM_BALACC"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CKPT_DIR / "best_by_val_balanced_acc.pt"
LAST_CKPT_PATH = CKPT_DIR / "last.pt"

PACKED_ROOT = SPLIT_ROOT / "PACKED_RUNS_FOR_BILSTM_BALACC"
for s in ["TRAIN", "VAL", "TEST"]:
    (PACKED_ROOT / s).mkdir(parents=True, exist_ok=True)

FEAT_ROOT = SPLIT_ROOT / "CNN_FEATURES_FOR_BILSTM_BALACC"
for s in ["TRAIN", "VAL", "TEST"]:
    (FEAT_ROOT / s).mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"

# First stable run: keep AMP off
USE_AMP = False

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / (1024**3), 2))
print("Torch:", torch.__version__)

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3
MODS = ["EEG", "ECG", "EMG", "MOV"]

BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4          
MAX_EPOCHS = 80
MIN_EPOCHS = 15
PATIENCE = 10
LR = 5e-4
WEIGHT_DECAY = 1e-4
CLIP_NORM = 0.5

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

RUN_CACHE_SIZE = 8
FEAT_BATCH_SIZE = 64

# =========================================================
# 3) Helpers
# =========================================================
def parse_base_from_filename(fname: str):
    for m in MODS:
        suf = f"_{m}.npz"
        if fname.endswith(suf):
            return fname[:-len(suf)]
    return None

def load_npz(path: Path):
    return np.load(path, allow_pickle=True)

def save_ckpt(path, model, optimizer, epoch, best_bal_acc):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optim": optimizer.state_dict() if optimizer is not None else None,
        "best_bal_acc": best_bal_acc,
    }, path)

def load_ckpt(path, model, optimizer=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and ckpt.get("optim", None) is not None:
        optimizer.load_state_dict(ckpt["optim"])
    return ckpt

# =========================================================
# 4) Pack split into one file per run
# =========================================================
def pack_split_runs(split_root: Path, packed_out_dir: Path):
    """
    Convert per-modality files into one packed file per run.

    Output keys:
      X_EEG, X_ECG, X_EMG, X_MOV, y, win_idx, base
    """
    packed_out_dir.mkdir(parents=True, exist_ok=True)
    eeg_dir = split_root / "EEG"
    if not eeg_dir.exists():
        raise RuntimeError(f"Missing EEG dir: {eeg_dir}")

    saved = 0
    skipped = 0

    eeg_files = sorted(eeg_dir.glob("*.npz"))
    if len(eeg_files) == 0:
        raise RuntimeError(f"No EEG files in {eeg_dir}")

    for eeg_path in eeg_files:
        base = parse_base_from_filename(eeg_path.name)
        if base is None:
            continue

        out_path = packed_out_dir / f"{base}.npz"
        if out_path.exists():
            skipped += 1
            continue

        mod_paths = {}
        ok = True
        for m in MODS:
            p = split_root / m / f"{base}_{m}.npz"
            if not p.exists():
                ok = False
                break
            mod_paths[m] = p
        if not ok:
            continue

        try:
            eeg_npz = load_npz(mod_paths["EEG"])
            ecg_npz = load_npz(mod_paths["ECG"])
            emg_npz = load_npz(mod_paths["EMG"])
            mov_npz = load_npz(mod_paths["MOV"])
        except Exception as e:
            print(f"⚠️ Failed to load run {base}: {e}")
            continue

        X_EEG = np.asarray(eeg_npz["X"], dtype=np.float32)
        X_ECG = np.asarray(ecg_npz["X"], dtype=np.float32)
        X_EMG = np.asarray(emg_npz["X"], dtype=np.float32)
        X_MOV = np.asarray(mov_npz["X"], dtype=np.float32)

        y_eeg = np.asarray(eeg_npz["y"]).astype(np.int8)
        y_ecg = np.asarray(ecg_npz["y"]).astype(np.int8)
        y_emg = np.asarray(emg_npz["y"]).astype(np.int8)
        y_mov = np.asarray(mov_npz["y"]).astype(np.int8)

        n = X_EEG.shape[0]

        if not (
            X_ECG.shape[0] == n and
            X_EMG.shape[0] == n and
            X_MOV.shape[0] == n and
            y_eeg.shape[0] == n and
            y_ecg.shape[0] == n and
            y_emg.shape[0] == n and
            y_mov.shape[0] == n
        ):
            print(f"⚠️ Skip {base}: inconsistent number of windows across modalities")
            continue

        if not (
            np.array_equal(y_eeg, y_ecg) and
            np.array_equal(y_eeg, y_emg) and
            np.array_equal(y_eeg, y_mov)
        ):
            print(f"⚠️ Skip {base}: y mismatch across modalities")
            continue

        X_EEG = np.nan_to_num(X_EEG, nan=0.0, posinf=0.0, neginf=0.0)
        X_ECG = np.nan_to_num(X_ECG, nan=0.0, posinf=0.0, neginf=0.0)
        X_EMG = np.nan_to_num(X_EMG, nan=0.0, posinf=0.0, neginf=0.0)
        X_MOV = np.nan_to_num(X_MOV, nan=0.0, posinf=0.0, neginf=0.0)

        if "win_idx" in eeg_npz:
            win_idx = np.asarray(eeg_npz["win_idx"], dtype=np.int32)
        else:
            win_idx = np.arange(n, dtype=np.int32)

        np.savez_compressed(
            out_path,
            X_EEG=X_EEG,
            X_ECG=X_ECG,
            X_EMG=X_EMG,
            X_MOV=X_MOV,
            y=y_eeg,
            win_idx=win_idx,
            base=np.array([base], dtype=object)
        )
        saved += 1

    print(f"✅ Packed {split_root.name}: saved={saved}, skipped_existing={skipped}, out={packed_out_dir}")

# =========================================================
# 5) Compute class weights from packed TRAIN only
# =========================================================
def compute_class_weights_from_packed_train(packed_train_root: Path):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    for p in sorted(packed_train_root.glob("*.npz")):
        try:
            npz = load_npz(p)
            y = np.asarray(npz["y"]).astype(int)
        except Exception:
            continue

        if y.size == 0:
            continue

        for c in range(NUM_CLASSES):
            counts[c] += int(np.sum(y == c))

    counts = np.maximum(counts, 1)
    total = counts.sum()

    weights = total / (NUM_CLASSES * counts.astype(np.float64))
    weights = weights / weights.mean()

    return counts, weights.astype(np.float32)

# =========================================================
# 6) Dataset
# =========================================================
class PackedRunDataset(Dataset):
    """
    Each sample = one window
    Underlying storage = one packed file per run
    """
    def __init__(self, packed_root: Path, cache_size: int = RUN_CACHE_SIZE):
        self.packed_root = Path(packed_root)
        self.cache_size = int(cache_size)
        self.run_files = sorted(self.packed_root.glob("*.npz"))

        if len(self.run_files) == 0:
            raise RuntimeError(f"No packed files found in {self.packed_root}")

        self.run_items = []
        self.flat = []
        self.run_to_flat_indices = []

        for p in self.run_files:
            try:
                npz = np.load(p, allow_pickle=True)
                y = np.asarray(npz["y"])
                base = str(npz["base"][0]) if "base" in npz else p.stem
                n = int(y.shape[0])
            except Exception as e:
                print(f"⚠️ Skip packed file {p.name}: {e}")
                continue

            start_pos = len(self.flat)
            for wi in range(n):
                self.flat.append((len(self.run_items), wi))
            end_pos = len(self.flat)

            self.run_items.append({
                "path": p,
                "base": base,
                "n_wins": n,
            })
            self.run_to_flat_indices.append(list(range(start_pos, end_pos)))

        if len(self.flat) == 0:
            raise RuntimeError(f"No usable samples in {self.packed_root}")

        self.cache = OrderedDict()

    def __len__(self):
        return len(self.flat)

    def _load_run(self, run_idx: int):
        if run_idx in self.cache:
            self.cache.move_to_end(run_idx)
            return self.cache[run_idx]

        item = self.run_items[run_idx]
        npz = np.load(item["path"], allow_pickle=True)

        run_data = {
            "X_EEG": np.asarray(npz["X_EEG"], dtype=np.float32),
            "X_ECG": np.asarray(npz["X_ECG"], dtype=np.float32),
            "X_EMG": np.asarray(npz["X_EMG"], dtype=np.float32),
            "X_MOV": np.asarray(npz["X_MOV"], dtype=np.float32),
            "y": np.asarray(npz["y"], dtype=np.int8),
            "win_idx": np.asarray(npz["win_idx"], dtype=np.int32),
            "base": str(npz["base"][0]) if "base" in npz else item["base"],
        }

        self.cache[run_idx] = run_data
        self.cache.move_to_end(run_idx)

        while len(self.cache) > self.cache_size:
            self.cache.popitem(last=False)

        return run_data

    def __getitem__(self, idx):
        run_idx, wi = self.flat[idx]
        run = self._load_run(run_idx)

        Xd = {
            "EEG": torch.from_numpy(run["X_EEG"][wi]),
            "ECG": torch.from_numpy(run["X_ECG"][wi]),
            "EMG": torch.from_numpy(run["X_EMG"][wi]),
            "MOV": torch.from_numpy(run["X_MOV"][wi]),
        }
        y = torch.tensor(int(run["y"][wi]), dtype=torch.long)
        base = run["base"]
        win_idx = torch.tensor(int(run["win_idx"][wi]), dtype=torch.long)

        return Xd, y, base, win_idx

# =========================================================
# 7) Run-aware batch sampler
# =========================================================
class RunChunkBatchSampler(Sampler):
    """
    Builds batches from windows belonging to the same run.
    """
    def __init__(self, dataset: PackedRunDataset, batch_size: int, shuffle: bool, drop_last: bool):
        self.dataset = dataset
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.drop_last = bool(drop_last)

    def __iter__(self):
        rng = np.random.default_rng()
        run_ids = np.arange(len(self.dataset.run_items))

        if self.shuffle:
            rng.shuffle(run_ids)

        all_batches = []
        for rid in run_ids:
            indices = self.dataset.run_to_flat_indices[rid].copy()
            if self.shuffle:
                rng.shuffle(indices)

            for i in range(0, len(indices), self.batch_size):
                batch = indices[i:i+self.batch_size]
                if len(batch) < self.batch_size and self.drop_last:
                    continue
                all_batches.append(batch)

        if self.shuffle:
            rng.shuffle(all_batches)

        for batch in all_batches:
            yield batch

    def __len__(self):
        total = 0
        for rid in range(len(self.dataset.run_items)):
            n = len(self.dataset.run_to_flat_indices[rid])
            if self.drop_last:
                total += n // self.batch_size
            else:
                total += math.ceil(n / self.batch_size)
        return total

# =========================================================
# 8) Collate
# =========================================================
def collate_mm(batch):
    Xs = {m: [] for m in MODS}
    ys = []
    bases = []
    win_idxs = []

    for Xd, y, base, wi in batch:
        for m in MODS:
            Xs[m].append(Xd[m])
        ys.append(y)
        bases.append(base)
        win_idxs.append(wi)

    Xs = {m: torch.stack(Xs[m], dim=0) for m in MODS}   # (B,C,T)
    ys = torch.stack(ys, dim=0)
    win_idxs = torch.stack(win_idxs, dim=0)
    return Xs, ys, bases, win_idxs

# =========================================================
# 9) Model
# =========================================================
class Conv1DEncoder(nn.Module):
    def __init__(self, in_ch: int, emb_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )
        self.proj = nn.Linear(256, emb_dim)

    def forward(self, x):
        h = self.net(x)            # (B,256,T')
        h = h.mean(dim=-1)         # GAP -> (B,256)
        z = self.proj(h)           # (B,emb_dim)
        return z

class MultiModalCNN(nn.Module):
    def __init__(self, ch_eeg, ch_ecg, ch_emg, ch_mov, emb_dim=128, num_classes=3):
        super().__init__()
        self.enc_eeg = Conv1DEncoder(ch_eeg, emb_dim)
        self.enc_ecg = Conv1DEncoder(ch_ecg, emb_dim)
        self.enc_emg = Conv1DEncoder(ch_emg, emb_dim)
        self.enc_mov = Conv1DEncoder(ch_mov, emb_dim)

        fused_dim = emb_dim * 4
        self.fuse = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.2),
            nn.Linear(fused_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, Xdict, return_features=False):
        z_eeg = self.enc_eeg(Xdict["EEG"])
        z_ecg = self.enc_ecg(Xdict["ECG"])
        z_emg = self.enc_emg(Xdict["EMG"])
        z_mov = self.enc_mov(Xdict["MOV"])

        z = torch.cat([z_eeg, z_ecg, z_emg, z_mov], dim=1)   # (B,4*emb)
        feat = self.fuse(z)                                  # (B,256)
        logits = self.classifier(feat)                       # (B,3)

        return logits if not return_features else (logits, feat)

# =========================================================
# 10) Utilities
# =========================================================
def get_channel_counts_from_packed_one_file(packed_root: Path):
    p = next(packed_root.glob("*.npz"), None)
    if p is None:
        raise RuntimeError(f"No packed files in {packed_root}")

    npz = np.load(p, allow_pickle=True)
    ch = {
        "EEG": int(npz["X_EEG"].shape[1]),
        "ECG": int(npz["X_ECG"].shape[1]),
        "EMG": int(npz["X_EMG"].shape[1]),
        "MOV": int(npz["X_MOV"].shape[1]),
    }
    return ch

# =========================================================
# 11) Train / Eval
# =========================================================
@torch.no_grad()
def evaluate(model, loader, device, loss_fn):
    model.eval()
    all_y = []
    all_p = []
    total_loss = 0.0
    n = 0

    for Xd, y, _, _ in loader:
        Xd = {k: v.to(device, non_blocking=True) for k, v in Xd.items()}
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(Xd)
            loss = loss_fn(logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError("Validation loss became NaN or Inf.")

        total_loss += float(loss.item()) * y.size(0)
        n += y.size(0)

        pred = torch.argmax(logits, dim=1)
        all_y.append(y.detach().cpu().numpy())
        all_p.append(pred.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    avg_loss = total_loss / max(1, n)

    return avg_loss, macro_f1, bal_acc

def train_one_epoch(model, loader, optimizer, scaler, device, loss_fn, accum_steps=1, clip_norm=1.0):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    n = 0
    steps_since_update = 0

    for step, (Xd, y, _, _) in enumerate(loader):
        Xd = {k: v.to(device, non_blocking=True) for k, v in Xd.items()}
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(Xd)
            loss = loss_fn(logits, y) / accum_steps

        if not torch.isfinite(loss):
            print("❌ Non-finite loss detected in training batch.")
            print("labels unique:", torch.unique(y))
            print("logits min/max:", logits.min().item(), logits.max().item())
            raise RuntimeError("Training loss became NaN or Inf.")

        scaler.scale(loss).backward()
        steps_since_update += 1

        if steps_since_update == accum_steps:
            scaler.unscale_(optimizer)
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            steps_since_update = 0

        total_loss += float(loss.item()) * y.size(0) * accum_steps
        n += y.size(0)

    if steps_since_update > 0:
        scaler.unscale_(optimizer)
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / max(1, n)

# =========================================================
# 12) Feature Extraction (BiLSTM-ready)
# =========================================================
@torch.no_grad()
def extract_features_per_run(model, packed_split_root: Path, out_dir: Path, batch_size: int):
    """
    Output per run:
      F       -> (n_wins, 256)
      y       -> (n_wins,)
      win_idx -> (n_wins,)
      base    -> object array with base string
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    model.eval()

    packed_files = sorted(packed_split_root.glob("*.npz"))
    if len(packed_files) == 0:
        raise RuntimeError(f"No packed files in {packed_split_root}")

    saved = 0

    for p in packed_files:
        run = np.load(p, allow_pickle=True)

        X_EEG = np.asarray(run["X_EEG"], dtype=np.float32)
        X_ECG = np.asarray(run["X_ECG"], dtype=np.float32)
        X_EMG = np.asarray(run["X_EMG"], dtype=np.float32)
        X_MOV = np.asarray(run["X_MOV"], dtype=np.float32)
        y     = np.asarray(run["y"], dtype=np.int8)
        win_idx = np.asarray(run["win_idx"], dtype=np.int32)
        base = str(run["base"][0]) if "base" in run else p.stem

        n = y.shape[0]
        feat_chunks = []

        for s in range(0, n, batch_size):
            e = min(s + batch_size, n)

            Xd = {
                "EEG": torch.from_numpy(X_EEG[s:e]).to(DEVICE, non_blocking=True),
                "ECG": torch.from_numpy(X_ECG[s:e]).to(DEVICE, non_blocking=True),
                "EMG": torch.from_numpy(X_EMG[s:e]).to(DEVICE, non_blocking=True),
                "MOV": torch.from_numpy(X_MOV[s:e]).to(DEVICE, non_blocking=True),
            }

            with torch.amp.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
                _, feat = model(Xd, return_features=True)

            if not torch.isfinite(feat).all():
                raise RuntimeError(f"Non-finite features detected during extraction for run {base}")

            feat_chunks.append(feat.detach().float().cpu().numpy())

        F_out = np.concatenate(feat_chunks, axis=0).astype(np.float32)

        out_path = out_dir / f"{base}.npz"
        np.savez_compressed(
            out_path,
            F=F_out,
            y=y,
            win_idx=win_idx,
            base=np.array([base], dtype=object)
        )
        saved += 1

    print(f"✅ Features saved: {saved} runs -> {out_dir}")

# =========================================================
# 13) Main
# =========================================================
def main():
    # -----------------------------------------------------
    # Step A: pack all splits once
    # -----------------------------------------------------
    print("\n[1/4] Packing TRAIN...")
    pack_split_runs(TRAIN_ROOT, PACKED_ROOT / "TRAIN")

    print("\n[1/4] Packing VAL...")
    pack_split_runs(VAL_ROOT, PACKED_ROOT / "VAL")

    print("\n[1/4] Packing TEST...")
    pack_split_runs(TEST_ROOT, PACKED_ROOT / "TEST")

    # -----------------------------------------------------
    # Step B: datasets and loaders
    # -----------------------------------------------------
    print("\n[2/4] Building datasets/loaders...")
    train_ds = PackedRunDataset(PACKED_ROOT / "TRAIN", cache_size=RUN_CACHE_SIZE)
    val_ds   = PackedRunDataset(PACKED_ROOT / "VAL",   cache_size=RUN_CACHE_SIZE)

    train_batch_sampler = RunChunkBatchSampler(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_batch_sampler = RunChunkBatchSampler(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False
    )

    train_loader = DataLoader(
        train_ds,
        batch_sampler=train_batch_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_mm,
    )

    val_loader = DataLoader(
        val_ds,
        batch_sampler=val_batch_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_mm,
    )

    # -----------------------------------------------------
    # Step C: model + loss + optimizer
    # -----------------------------------------------------
    counts, w = compute_class_weights_from_packed_train(PACKED_ROOT / "TRAIN")
    print("TRAIN label counts:", {i: int(counts[i]) for i in range(NUM_CLASSES)})
    print("Class weights:", w.tolist())

    class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=class_w)

    ch = get_channel_counts_from_packed_one_file(PACKED_ROOT / "TRAIN")
    print("Channel counts:", ch)

    model = MultiModalCNN(
        ch_eeg=ch["EEG"],
        ch_ecg=ch["ECG"],
        ch_emg=ch["EMG"],
        ch_mov=ch["MOV"],
        emb_dim=128,
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
    )

    Xd0, y0, _, _ = next(iter(train_loader))
    Xd0 = {k: v.to(DEVICE) for k, v in Xd0.items()}
    y0 = y0.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits0 = model(Xd0)
        print("Sanity logits min/max:", logits0.min().item(), logits0.max().item())
        print("Sanity logits finite:", torch.isfinite(logits0).all().item())

    # -----------------------------------------------------
    # Step D: train
    # -----------------------------------------------------
    print("\n[3/4] Training...")
    best_bal_acc = -1.0
    best_epoch = -1
    bad_epochs = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            loss_fn=loss_fn,
            accum_steps=GRAD_ACCUM_STEPS,
            clip_norm=CLIP_NORM
        )

        val_loss, val_f1, val_bal_acc = evaluate(
            model=model,
            loader=val_loader,
            device=DEVICE,
            loss_fn=loss_fn
        )

        scheduler.step(val_bal_acc)
        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:03d} | "
            f"lr {lr_now:.2e} | "
            f"train_loss {train_loss:.4f} | "
            f"val_loss {val_loss:.4f} | "
            f"val_bal_acc {val_bal_acc:.4f} | "
            f"val_macro_f1 {val_f1:.4f}"
        )

        save_ckpt(LAST_CKPT_PATH, model, optimizer, epoch, best_bal_acc)

        improved = val_bal_acc > (best_bal_acc + 1e-6)
        if improved:
            best_bal_acc = val_bal_acc
            best_epoch = epoch
            bad_epochs = 0
            save_ckpt(BEST_CKPT_PATH, model, optimizer, epoch, best_bal_acc)
            print(f"✅ Saved BEST checkpoint (val_bal_acc={best_bal_acc:.4f})")
        else:
            bad_epochs += 1

        if epoch >= MIN_EPOCHS and bad_epochs >= PATIENCE:
            print(f"🛑 Early stopping at epoch {epoch}. Best epoch={best_epoch}, val_bal_acc={best_bal_acc:.4f}")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\nTraining done.")
    print("Best checkpoint:", BEST_CKPT_PATH)
    print("Best val_bal_acc:", best_bal_acc, "at epoch", best_epoch)

    # -----------------------------------------------------
    # Step E: load best and extract features
    # -----------------------------------------------------
    print("\n[4/4] Loading BEST checkpoint for feature extraction...")
    _ = load_ckpt(BEST_CKPT_PATH, model, optimizer=None, map_location=DEVICE)

    print("\nExtracting TRAIN features...")
    extract_features_per_run(
        model=model,
        packed_split_root=PACKED_ROOT / "TRAIN",
        out_dir=FEAT_ROOT / "TRAIN",
        batch_size=FEAT_BATCH_SIZE
    )

    print("\nExtracting VAL features...")
    extract_features_per_run(
        model=model,
        packed_split_root=PACKED_ROOT / "VAL",
        out_dir=FEAT_ROOT / "VAL",
        batch_size=FEAT_BATCH_SIZE
    )

    print("\nExtracting TEST features...")
    extract_features_per_run(
        model=model,
        packed_split_root=PACKED_ROOT / "TEST",
        out_dir=FEAT_ROOT / "TEST",
        batch_size=FEAT_BATCH_SIZE
    )

    print("\n✅ All done.")
    print("Packed root   :", PACKED_ROOT)
    print("Features root :", FEAT_ROOT)

if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM (GB): 7.96
Torch: 2.10.0+cu128

[1/4] Packing TRAIN...
✅ Packed TRAIN: saved=0, skipped_existing=2316, out=C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\PACKED_RUNS_FOR_BILSTM_BALACC\TRAIN

[1/4] Packing VAL...
✅ Packed VAL: saved=0, skipped_existing=190, out=C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\PACKED_RUNS_FOR_BILSTM_BALACC\VAL

[1/4] Packing TEST...
✅ Packed TEST: saved=0, skipped_existing=344, out=C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\PACKED_RUNS_FOR_BILSTM_BALACC\TEST

[2/4] Building datasets/loaders...
TRAIN label counts: {0: 1718712, 1: 129702, 2: 11090}
Class weights: [0.017727376893162727, 0.23490968346595764, 2.7473628520965576]
Channel counts: {'EEG': 2, 'ECG': 1, 'EMG': 1, 'MOV': 4}
Sanity logits min/max: -0.07692092657089233 0.1953006386756897
Sanity logits finite: True

[3/4] Training...
Epoch 001 | lr 5.00e-04 | train_loss 0.3690 | val_loss 0.6112 | val_b

In [14]:
from pathlib import Path
import torch
import numpy as np

# =========================================================
# Paths
# =========================================================
SPLIT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD")

CKPT_DIR = SPLIT_ROOT / "CNN_CKPTS_FOR_BILSTM_BALACC"
BEST_CKPT_PATH = CKPT_DIR / "best_by_val_balanced_acc.pt"
LAST_CKPT_PATH = CKPT_DIR / "last.pt"

FEAT_ROOT = SPLIT_ROOT / "CNN_FEATURES_FOR_BILSTM_BALACC"
TRAIN_FEAT = FEAT_ROOT / "TRAIN"
VAL_FEAT   = FEAT_ROOT / "VAL"
TEST_FEAT  = FEAT_ROOT / "TEST"

# =========================================================
# Show checkpoint info
# =========================================================
def show_ckpt_info(path: Path, name: str):
    if not path.exists():
        print(f"❌ {name} checkpoint not found: {path}")
        return

    ckpt = torch.load(path, map_location="cpu")

    print("\n" + "="*50)
    print(f"{name} CHECKPOINT")
    print("="*50)
    print("Path           :", path)
    print("Saved epoch    :", ckpt.get("epoch", "N/A"))
    print("Best val balacc:", ckpt.get("best_bal_acc", "N/A"))

    model_state = ckpt.get("model", {})
    if isinstance(model_state, dict):
        print("Model params   :", len(model_state))

    optim_state = ckpt.get("optim", None)
    print("Has optimizer  :", optim_state is not None)

# =========================================================
# Show feature files summary
# =========================================================
def summarize_feature_split(split_dir: Path, split_name: str):
    files = sorted(split_dir.glob("*.npz"))

    print("\n" + "="*50)
    print(f"{split_name} FEATURES")
    print("="*50)

    if len(files) == 0:
        print("No feature files found.")
        return

    print("Number of run files:", len(files))

    total_windows = 0
    feat_dim = None
    label_counts = {}

    for p in files:
        data = np.load(p, allow_pickle=True)

        F = np.asarray(data["F"])
        y = np.asarray(data["y"])

        total_windows += len(y)

        if feat_dim is None and F.ndim == 2:
            feat_dim = F.shape[1]

        unique, counts = np.unique(y, return_counts=True)
        for cls, cnt in zip(unique, counts):
            label_counts[int(cls)] = label_counts.get(int(cls), 0) + int(cnt)

    print("Total windows  :", total_windows)
    print("Feature dim    :", feat_dim)
    print("Label counts   :", label_counts)

    # preview first file
    first_file = files[0]
    first_data = np.load(first_file, allow_pickle=True)
    print("\nExample file   :", first_file.name)
    print("F shape        :", np.asarray(first_data["F"]).shape)
    print("y shape        :", np.asarray(first_data["y"]).shape)
    print("win_idx shape  :", np.asarray(first_data["win_idx"]).shape)
    print("base           :", str(first_data["base"][0]) if "base" in first_data else first_file.stem)

# =========================================================
# Run
# =========================================================
show_ckpt_info(BEST_CKPT_PATH, "BEST")
show_ckpt_info(LAST_CKPT_PATH, "LAST")

summarize_feature_split(TRAIN_FEAT, "TRAIN")
summarize_feature_split(VAL_FEAT, "VAL")
summarize_feature_split(TEST_FEAT, "TEST")


BEST CHECKPOINT
Path           : C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\CNN_CKPTS_FOR_BILSTM_BALACC\best_by_val_balanced_acc.pt
Saved epoch    : 23
Best val balacc: 0.37401704712595857
Model params   : 86
Has optimizer  : True

LAST CHECKPOINT
Path           : C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\CNN_CKPTS_FOR_BILSTM_BALACC\last.pt
Saved epoch    : 33
Best val balacc: 0.37401704712595857
Model params   : 86
Has optimizer  : True

TRAIN FEATURES
Number of run files: 2316
Total windows  : 1859504
Feature dim    : 256
Label counts   : {0: 1718712, 1: 129702, 2: 11090}

Example file   : sub-001_ses-01_run-01_win4s_pre900s.npz
F shape        : (900, 256)
y shape        : (900,)
win_idx shape  : (900,)
base           : sub-001_ses-01_run-01_win4s_pre900s

VAL FEATURES
Number of run files: 190
Total windows  : 155543
Feature dim    : 256
Label counts   : {1: 14188, 2: 841, 0: 140514}

Example file   : sub-017_ses-01_run-01_win4s_pre900s.npz
F shape    